In [1]:
# ==================================================================
# ██████  CELL 1: INSTALL (paste this into first Colab cell)  ██████
# ==================================================================
# %%capture
# !pip install unsloth
# !pip install accelerate==0.33.0
# !pip install --upgrade --no-deps trl


# ==================================================================
# ██████  CELL 2: TRAIN (paste this into second Colab cell)   ██████
# ==================================================================
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# ──────────────────────────────────────────────────────────────
# FIX: Monkey-patch accelerate to handle NoneType device index
# This fixes the "device() received NoneType" crash in accelerate >= 0.34
# Root cause: 4-bit models report None as device index for some params
# Fix: Replace None with 0 (cuda:0) in the device map before prepare_model
# ──────────────────────────────────────────────────────────────
import accelerate
_original_prepare_model = accelerate.Accelerator.prepare_model

def _patched_prepare_model(self, model, device_placement=None, evaluation_mode=False):
    # Fix None device indices in the model's device map
    if hasattr(model, 'hf_device_map'):
        for key in model.hf_device_map:
            if model.hf_device_map[key] is None:
                model.hf_device_map[key] = 0
    return _original_prepare_model(self, model, device_placement, evaluation_mode)

accelerate.Accelerator.prepare_model = _patched_prepare_model
print("✅ Accelerate monkey-patch applied")

# --- Config ---
max_seq_length = 1024  # Our samples are short Q&A — no need for 2048
load_in_4bit = True

# --- Load Model ---
print("Loading Qwen3-8B (4-bit)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3-8B",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = load_in_4bit,
)

# --- Apply LoRA ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# --- Dataset ---
dataset = load_dataset("json", data_files="julia_microtune_combined.jsonl", split="train")

alpaca_prompt = """### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = alpaca_prompt.format(instruction, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched=True)

# --- Trainer ---
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

# --- Train! ---
print(f"Dataset: {len(dataset)} samples")
print("Starting training...")
trainer_stats = trainer.train()
print(f"Training done! Final loss: {trainer_stats.training_loss:.4f}")

# --- Save Adapter ---
model.save_pretrained("julia_microtune_adapter")
tokenizer.save_pretrained("julia_microtune_adapter")
print("Adapter saved to julia_microtune_adapter/")




# ==================================================================
# ██████  CELL 3: ZIP & DOWNLOAD (paste into third Colab cell) █████
# ==================================================================
# import shutil
# from google.colab import files
# shutil.make_archive("julia_microtune_adapter", "zip", "julia_microtune_adapter")
# files.download("julia_microtune_adapter.zip")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Accelerate monkey-patch applied
Loading Qwen3-8B (4-bit)...
==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.2.1 patched 36 layers with 36 QKV layers, 36 O layers and 0 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/1511 [00:00<?, ? examples/s]

Dataset: 1511 samples
Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,511 | Num Epochs = 2 | Total steps = 190
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 15,335,424 of 8,206,070,784 (0.19% trained)


Step,Training Loss
10,2.320500
20,2.046600
30,1.914600
40,1.918600
50,1.880800
60,1.891600
70,1.794500
80,1.867400
90,1.787300
100,1.777800


Training done! Final loss: 1.8311
Adapter saved to julia_microtune_adapter/
